In [ ]:
%load_ext cudf.pandas

In [ ]:
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import numpy as np

In [ ]:
%%time
### cell 0 ###

df1 = pd.read_csv(
    "/home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/input/tmdb-movie-metadata/tmdb_5000_credits.csv"
)
df2 = pd.read_csv(
    "/home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/input/tmdb-movie-metadata/tmdb_5000_movies.csv"
)
factor = 50
df1 = pd.concat([df1] * factor, ignore_index=True)
df1.info()

In [ ]:
%%time
### cell 1 ###

df1.columns = ["id", "tittle", "cast", "crew"]
df2 = df2.merge(df1, on="id")

In [ ]:
%%time
### cell 2 ###

df2.head(5)

In [ ]:
%%time
### cell 3 ###

C = df2["vote_average"].mean()
C

In [ ]:
%%time
### cell 4 ###

m = df2["vote_count"].quantile(0.9)
m

In [ ]:
%%time
### cell 5 ###

q_movies = df2.loc[df2["vote_count"] >= m].copy()
q_movies.shape

In [ ]:
%%time
### cell 6 ###

# Vectorized computation of weighted rating to leverage GPU
den = q_movies["vote_count"] + m
q_movies["score"] = (
    q_movies["vote_count"] / den * q_movies["vote_average"] + m / den * C
)

In [ ]:
%%time
### cell 7 ###

q_movies = q_movies.nlargest(10, "score")[
    ["title", "vote_count", "vote_average", "score"]
]
q_movies

In [ ]:
%%time
### cell 8 ###

pop = df2.sort_values("popularity", ascending=False)

In [ ]:
%%time
### cell 9 ###

df2["overview"].head(5)

In [ ]:
%%time
### cell 10 ###

# Import TfIdfVectorizer from scikit-learn
df2["overview"] = df2["overview"].fillna("")

In [ ]:
%%time
### cell 11 ###

# Parse the stringified features into their corresponding python objects
import json

features = ["cast", "crew", "keywords", "genres"]
for feature in features:
    df2[feature] = df2[feature].apply(json.loads)
# for feature in features:
#     df2[feature] = df2[feature].apply(literal_eval)

In [ ]:
%%time
### cell 12 ###


# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i["job"] == "Director":
            return i["name"]
    return np.nan


def get_list(x):
    if isinstance(x, list):
        names = [i["name"] for i in x]
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []


# Define new director, cast, genres and keywords features that are in a suitable form.
df2["director"] = df2["crew"].apply(get_director)

features = ["cast", "keywords", "genres"]
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [ ]:
%%time
### cell 13 ###

# Print the new features of the first 3 films
df2[["title", "cast", "director", "keywords", "genres"]].head(3)

In [ ]:
%%time
### cell 14 ###


# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ""


# Apply clean_data function to your features.
features = ["cast", "keywords", "director", "genres"]

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

In [ ]:
%%time
### cell 15 ###

df2["keywords"] = (
    df2["keywords"]
    .astype(str)
    .str.replace(r"[\[\]']", "", regex=True)
    .str.replace(", ", " ", regex=False)
)
df2["cast"] = (
    df2["cast"]
    .astype(str)
    .str.replace(r"[\[\]']", "", regex=True)
    .str.replace(", ", " ", regex=False)
)
df2["genres"] = (
    df2["genres"]
    .astype(str)
    .str.replace(r"[\[\]']", "", regex=True)
    .str.replace(", ", " ", regex=False)
)

# Concatenate all features into the 'soup' column using GPU string concatenation
df2["soup"] = (
    df2["keywords"] + " " + df2["cast"] + " " + df2["director"] + " " + df2["genres"]
)